In [2]:
import numpy as np
import matplotlib.pyplot as plt
import xicsrt

import sys
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirxics_jax")
import xics_jax

from xicsrt.tools import xicsrt_multi_voigt

config = {}

config['general'] = {}
config['general']['number_of_iter'] = 1
config['general']['save_images'] = False
config['general']['save_results'] = False

config['sources'] = {}
config['sources']['source'] = {
    'class_name': 'XicsrtSourceGeneric',

    'origin': [0.0, 0.0, 0.0],
    'zaxis': [0.0, 0.0, 1.0],

    'intensity': 10000,
    'spatial_dist': 'uniform',
    'xsize': 0.0,
    'ysize': 0.0,
    'zsize': 0.0,

    'angular_dist': 'isotropic',
    'spread': 0.01,

    'wavelength_dist': 'ar16_voigt',

    'ar16_ti': 2.0,
    'ar16_te': 3.0,
    'ar16_scale_factor': 1.0,    

     'multi_gridsize': None,
    'multi_cutoff': 1e-4,
}

config['optics'] = {}

config['optics']['detector'] = {
    'class_name': 'XicsrtOpticDetector',
    'origin': [0.0, 0.0, 1.0],
    'zaxis': [0.0, 0.0, -1.0],
    'xsize': 0.2,
    'ysize': 0.2,
    'pixel_size': 0.001,
}

# testing _XicsrtSourceGeneric.py implementation of 'wavelength_range'
config["sources"]["plasma"] = {}
config["sources"]["plasma"]["ar16_wavelength_range"] = [3.94, 4.0]

results = xicsrt.raytrace(config)

source_rays = results['found']['history']['source']
detector_rays = results['found']['history']['detector']

print("Generated rays:", len(source_rays['wavelength']))
print("Detector rays:", len(detector_rays['wavelength']))
print("Wavelength min/max:", np.min(source_rays['wavelength']), np.max(source_rays['wavelength']))

import xicsrt.visual.xicsrt_2d__matplotlib as xicsrt_2d
fig = xicsrt_2d.plot_intersect(results, 'detector', aspect='equal')

import xicsrt.visual.xicsrt_3d__plotly as xicsrt_3d

fig = xicsrt_3d.figure()
# xicsrt_3d.add_fluxsurfaces(result['config'], range_n=(0, np.pi*2))
xicsrt_3d.add_rays(results)
xicsrt_3d.add_optics(results['config'])
xicsrt_3d.add_sources(results['config'])

fig.update_layout(
    autosize=False,
    width=1000,
    height=800,
)

xicsrt_3d.show()

plt.hist(source_rays['wavelength'], bins=200)
plt.xlabel("Wavelength [Å]")
plt.ylabel("Counts")
plt.title("Sampled multiline Voigt wavelengths")
plt.show()

# filter out rays above some value (4.01) and set wavelength range
# filter out beryllium lines or certain classes of lines

 INFO:xicsrt: Starting run: 1 of 1
 INFO:xicsrt: Seeding np.random with None


Exception: User option not recognized: ar16_ti

In [ ]:
%matplotlib widget
import numpy as np
from collections import OrderedDict
import os

# Setup the module path.
import sys
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_contrib")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_analysis")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirfusion_library")

## Start Logging
import logging
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('PIL').setLevel(logging.WARNING)
logging.getLogger('h5py').setLevel(logging.WARNING)
logging.getLogger('jax').setLevel(logging.WARNING)

from mirutil import mirprint

# Import xicsrt modules
import xicsrt
from xicsrt.util import profiler
from xicsrt.tools import xicsrt_multi_voigt

from w7x_npablant import xicsrt_w7x_npablant

In [ ]:
config = xicsrt_w7x_npablant.get_config()
config = xicsrt_w7x_npablant.initialize(config)

config['general']['comment'] = """
W7-X run without velocity profile profile.
Based on Op1.2b geometry calibration.

Simple polynomial profiles for emissivity, temperature and velocity.

Full crystal W7-X Ar16+ raytracing simulation with realistic W7-X plasma geometry.
"""

config['general']['number_of_runs'] = 1
config['general']['number_of_iter'] = 1
config['general']['save_images'] = True
config['general']['make_directories'] = True

config['general']['keep_history'] = True

config['general']['output_path'] = r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026"
config['general']['output_suffix'] = '_ar16_voigt'

config['sources']['plasma']['wavelength_dist'] = 'ar16_voigt'

config['sources']['plasma']['wout_file'] = r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\wout.nc"
config['sources']['plasma']['enable_velocity'] = False
config['sources']['plasma']['enable_flux_compression'] = False
config['sources']['plasma']['time_resolution'] = 1e-3
config['sources']['plasma']['spread'] = np.radians(1.0)
config['sources']['plasma']['xsize'] = 0.15

In [ ]:
profiler.resetProfiler()
profiler.startProfiler()

result = xicsrt.raytrace(config)
    
profiler.stopProfiler()
#print('', flush=True)
profiler.report()

In [ ]:
import xicsrt.visual.xicsrt_3d__plotly as xicsrt_3d

fig = xicsrt_3d.figure()
# xicsrt_3d.add_fluxsurfaces(result['config'], range_n=(0, np.pi*2))
xicsrt_3d.add_rays(result)
xicsrt_3d.add_optics(result['config'])
xicsrt_3d.add_sources(result['config'])

fig.update_layout(
    autosize=False,
    width=1000,
    height=800,
)

xicsrt_3d.show()

In [ ]:
import xicsrt.visual.xicsrt_2d__matplotlib as xicsrt_2d
fig = xicsrt_2d.plot_intersect(result, 'crystal', aspect='equal')

In [ ]:
import xicsrt.visual.xicsrt_2d__matplotlib as xicsrt_2d
fig = xicsrt_2d.plot_intersect(result, 'detector', aspect='equal')

In [ ]:
import xicsrt.visual.xicsrt_2d__matplotlib as xicsrt_2d
fig = xicsrt_2d.plot_image(result, 'detector')

In [ ]:
from matplotlib import pyplot
pyplot.close('all')